In [68]:
# ==============================================================================
# [GLOBAL CELL 1] KONFIGURASI SENTRAL & SEEDING (PRUNING STRESS TEST)
# ==============================================================================
import os
import random
from enum import Enum
from typing import Tuple, List

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------------------------
# 1. ENUMERASI ARSITEKTUR (Standar 3 & 5)
# ------------------------------------------------------------------------------
class ModelArch(str, Enum):
    MOBILENET_V3 = 'MobileNetV3Small'
    EFFICIENTNET_B0 = 'EfficientNetB0'
    RESNET_50 = 'ResNet50'

class PruningMetric(str, Enum):
    L1_NORM = "l1_norm"
    L2_NORM = "l2_norm"

# ------------------------------------------------------------------------------
# 2. SENTRALISASI KONFIGURASI (Standar 2 & 4)
# ------------------------------------------------------------------------------
class PruningConfig:
    """Konfigurasi utama untuk Fase One-Shot Stress Test & Information Loss."""
    
    # Dataset & Dimensi (Hanya butuh Validation Set untuk Stress Test & Entropy)
    PATH_VAL_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/val"
    IMG_SIZE: Tuple[int, int] = (224, 224)
    VAL_BATCH_SIZE: int = 32 # Diseragamkan 32 agar aman untuk VRAM Kaggle
    
    # Hyperparameter Pruning (L2-Norm)
    EPSILON_NORM: float = 1e-9
    # PRUNING_RATIOS: List[float] = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    PRUNING_RATIOS: List[float] = [0.0, 0.5]
    PRUNING_METRIC: PruningMetric = PruningMetric.L2_NORM
    
    # Direktori Input (Sumber Model FP32 yang sudah ditraining di Notebook sebelumnya)
    # Sesuaikan path ini dengan dataset/output Kaggle tempat model training Anda berada
    PATH_TRAINED_MODELS: str = "/kaggle/input/models/marrioqqqq/prune-model-ffb/keras/default/1"
    
    # Direktori Output Pruning Dinamis (Standar 4)
    BASE_OUTPUT_DIR: str = "/kaggle/working/pruning_outputs"

    @classmethod
    def get_output_dir(cls, arch: ModelArch) -> str:
        target_dir = os.path.join(cls.BASE_OUTPUT_DIR, arch.value)
        os.makedirs(target_dir, exist_ok=True)
        return target_dir

    @classmethod
    def get_input_model_path(cls, arch: ModelArch) -> str:
        """Mengambil model .keras final FP32 dengan sistem Auto-Detect struktur Kaggle."""
        
        # Kemungkinan 1: Struktur diratakan (Flattened) oleh Kaggle Models
        path_flat = os.path.join(cls.PATH_TRAINED_MODELS, f"model_{arch.value}_phase2.keras")
        
        # Kemungkinan 2: Struktur folder dipertahankan oleh Kaggle Dataset
        path_nested = os.path.join(cls.PATH_TRAINED_MODELS, arch.value, f"model_{arch.value}_phase2.keras")
        
        if os.path.exists(path_flat):
            print(f"[DEBUG PATH] Model ditemukan di struktur Flat: {path_flat}")
            return path_flat
        elif os.path.exists(path_nested):
            print(f"[DEBUG PATH] Model ditemukan di struktur Nested: {path_nested}")
            return path_nested
        else:
            # Jika kedua path tidak ada, kembalikan flat path untuk error log
            return path_flat

    @classmethod
    def get_db_path(cls, arch: ModelArch) -> str:
        """Standar 8: Menggantikan file CSV macro & micro dengan satu database solid."""
        return os.path.join(cls.get_output_dir(arch), f"pruning_stresstest_{arch.value}.db")

# ------------------------------------------------------------------------------
# 3. REPRODUKSIBILITAS (Standar 3)
# ------------------------------------------------------------------------------
def set_deterministic_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"[SETUP] Deterministic Seed dikunci pada: {seed}")

set_deterministic_seed(42)
print(f"[SETUP] TensorFlow Version: {tf.__version__}")

[SETUP] Deterministic Seed dikunci pada: 42
[SETUP] TensorFlow Version: 2.20.0


In [69]:
# ==============================================================================
# [CELL 2] KERAS SERIALIZATION PATCH & SQLITE DATABASE ENGINE
# ==============================================================================
import sqlite3
from typing import Dict, Any
from tensorflow.keras import layers

# ------------------------------------------------------------------------------
# 1. KOREKSI SERIALISASI KERAS (MONKEY PATCHING)
# ------------------------------------------------------------------------------
def patch_keras_serialization() -> None:
    """
    Menghapus konfigurasi kuantisasi bawaan Keras saat model di-rebuild via from_config().
    Mencegah TypeError saat layer dipotong (pruned) secara fisik.
    """
    if hasattr(layers.Conv2D, '_is_patched_for_pruning'):
        return # Cegah patch berulang jika cell dijalankan dua kali
        
    _orig_dense = layers.Dense.__init__
    _orig_conv2d = layers.Conv2D.__init__
    _orig_dwconv2d = layers.DepthwiseConv2D.__init__
    _orig_bn = layers.BatchNormalization.__init__

    def safe_dense(self, *args, **kwargs): 
        kwargs.pop('quantization_config', None); _orig_dense(self, *args, **kwargs)
    def safe_conv2d(self, *args, **kwargs): 
        kwargs.pop('quantization_config', None); _orig_conv2d(self, *args, **kwargs)
    def safe_dwconv2d(self, *args, **kwargs): 
        kwargs.pop('quantization_config', None); _orig_dwconv2d(self, *args, **kwargs)
    def safe_bn(self, *args, **kwargs): 
        kwargs.pop('quantization_config', None); _orig_bn(self, *args, **kwargs)

    layers.Dense.__init__ = safe_dense
    layers.Conv2D.__init__ = safe_conv2d
    layers.DepthwiseConv2D.__init__ = safe_dwconv2d
    layers.BatchNormalization.__init__ = safe_bn
    layers.Conv2D._is_patched_for_pruning = True
    print("[SETUP] Keras Layer Serialization berhasil di-patch untuk Pruning.")

patch_keras_serialization()

# ------------------------------------------------------------------------------
# 2. SQLITE LOGGER BASE CLASS (Standar 8 & 9)
# ------------------------------------------------------------------------------
class PruningSQLiteLogger:
    """
    Mengelola logging eksperimen Stress Test ke dalam SQLite.
    Menggantikan output file mobnet_macro_stresstest.csv & micro_layerwise.csv.
    """
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._initialize_db()

    def _initialize_db(self) -> None:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Tabel Makro: Menyimpan dampak pruning terhadap model secara keseluruhan
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS macro_metrics (
                target_ratio_pct INTEGER PRIMARY KEY,
                params_active INTEGER,
                params_reduced_pct REAL,
                zero_shot_accuracy REAL,
                activation_entropy REAL
            )
        ''')
        
        # Tabel Mikro: Menyimpan detail pemotongan (Slicing) per layer
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS micro_metrics (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                target_ratio_pct INTEGER,
                block_name TEXT,
                original_filters INTEGER,
                pruned_filters INTEGER,
                surviving_filters INTEGER,
                actual_pruning_ratio REAL,
                l2_norm_energy_mean REAL,
                l2_norm_energy_max REAL
            )
        ''')
        conn.commit()
        conn.close()

    def log_macro(self, data: Dict[str, Any]) -> None:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT OR REPLACE INTO macro_metrics 
            (target_ratio_pct, params_active, params_reduced_pct, zero_shot_accuracy, activation_entropy)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            data['Target_Ratio_Pct'], data['Params_Active'], 
            data['Params_Reduced_Pct'], data['Zero_Shot_Accuracy'], data['Activation_Entropy']
        ))
        conn.commit()
        conn.close()

    def log_micro_batch(self, target_ratio: int, plans: Dict[str, Dict[str, Any]]) -> None:
        """Memasukkan data layer-wise secara massal (batch) agar I/O disk efisien."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        records = []
        for block_name, details in plans.items():
            records.append((
                target_ratio, block_name,
                details['original_filters'], details['pruned_filters'], details['surviving_filters'],
                float(details['ratio']), float(details['norm_energy_mean']), float(details['norm_energy_max'])
            ))
            
        cursor.executemany('''
            INSERT INTO micro_metrics 
            (target_ratio_pct, block_name, original_filters, pruned_filters, surviving_filters, 
             actual_pruning_ratio, l2_norm_energy_mean, l2_norm_energy_max)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', records)
        
        conn.commit()
        conn.close()

print("[BERHASIL] Keras Serialization Patch & SQLite Engine siap.")

[BERHASIL] Keras Serialization Patch & SQLite Engine siap.


In [70]:
# ==============================================================================
# [CELL 3] PRUNING ANALYZER: INFORMATION ENTROPY & NORM PLANNER (SCALABLE)
# ==============================================================================
import re
from enum import Enum
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from typing import Dict, Any

# ------------------------------------------------------------------------------
# 1. ENUMERASI METRIK PRUNING (EXTENSIBLE)
# ------------------------------------------------------------------------------
class PruningMetric(str, Enum):
    L1_NORM = "l1_norm"
    L2_NORM = "l2_norm"

class PruningAnalyzer:
    """
    Standar 5 & 9: Modul Analitik Agnostik dengan dukungan multi-metrik.
    Kini mendukung transisi mulus antara L1-Norm dan L2-Norm.
    """
    def __init__(self, config: PruningConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture

    # --------------------------------------------------------------------------
    # 2. ANALISIS ENTROPI INFORMASI
    # --------------------------------------------------------------------------
    def compute_activation_entropy(self, model: tf.keras.Model, dataset: tf.data.Dataset) -> float:
        target_layer = None
        for layer in reversed(model.layers):
            if isinstance(layer, layers.Conv2D):
                target_layer = layer
                break
                
        if target_layer is None: 
            return 0.0

        activation_model = tf.keras.Model(inputs=model.input, outputs=target_layer.output)
        entropies = []
        
        for batch, _ in dataset.take(1):
            acts = activation_model(batch, training=False)
            acts_flat = tf.reshape(acts, [tf.shape(acts)[0], -1])
            p = tf.nn.softmax(acts_flat, axis=-1)
            entropy = -tf.reduce_sum(p * tf.math.log(p + 1e-9), axis=-1)
            entropies.append(float(tf.reduce_mean(entropy)))
            
        return float(np.mean(entropies))

    # --------------------------------------------------------------------------
    # 3. KALKULATOR ENERGI (STRATEGY PATTERN)
    # --------------------------------------------------------------------------
    def _calculate_energy(self, weights: np.ndarray, metric: PruningMetric, axis: tuple) -> np.ndarray:
        """Fungsi modular untuk menghitung skor filter. Tambahkan metrik baru di sini."""
        if metric == PruningMetric.L2_NORM:
            return np.sum(np.square(weights), axis=axis)
        elif metric == PruningMetric.L1_NORM:
            return np.sum(np.abs(weights), axis=axis)
        else:
            raise ValueError(f"[ERROR] Metrik {metric.value} belum didukung.")

    # --------------------------------------------------------------------------
    # 4. NORM PLANNER (METRIC AGNOSTIC)
    # --------------------------------------------------------------------------
    def plan_normalized(self, model: tf.keras.Model, target_ratio: float, metric: PruningMetric = PruningMetric.L2_NORM) -> Dict[str, Dict[str, Any]]:
        """Mengekstrak rencana mask menggunakan metrik yang dipilih (Default: L2-Norm)."""
        blocks: Dict[str, Dict[str, tf.keras.layers.Layer]] = {}
        
        regex_pattern = r'(_expand_conv|_dwconv|_se_reduce|_se_expand|_project_conv)' if self.arch == ModelArch.EFFICIENTNET_B0 else r'(_expand|_depthwise|_squeeze|_project)'
        
        for layer in model.layers:
            match = re.split(regex_pattern, layer.name)
            if len(match) > 1:
                prefix = match[0]
                if prefix not in blocks: blocks[prefix] = {}
                
                if ("_expand" in layer.name or "_expand_conv" in layer.name) and "bn" not in layer.name: 
                    blocks[prefix]['expand'] = layer
                elif ("_depthwise" in layer.name or "_dwconv" in layer.name) and "bn" not in layer.name: 
                    blocks[prefix]['depthwise'] = layer
                elif ("project" in layer.name or "_project_conv" in layer.name) and "bn" not in layer.name: 
                    blocks[prefix]['project'] = layer

        plans: Dict[str, Dict[str, Any]] = {}
        for prefix, components in blocks.items():
            if 'expand' not in components: continue
            
            w_expand = components['expand'].get_weights()
            if not w_expand: continue
            
            num_filters: int = w_expand[0].shape[-1]
            
            # --- PENGGUNAAN METRIK DINAMIS ---
            raw_energy = self._calculate_energy(w_expand[0], metric, axis=(0, 1, 2))
            norm_energy = raw_energy / (np.max(raw_energy) + self.cfg.EPSILON_NORM)
            
            if 'depthwise' in components and components['depthwise'].get_weights():
                dw_energy = self._calculate_energy(components['depthwise'].get_weights()[0], metric, axis=(0, 1, 3))
                norm_energy += (dw_energy / (np.max(dw_energy) + self.cfg.EPSILON_NORM))
                    
            if 'project' in components and components['project'].get_weights():
                proj_energy = self._calculate_energy(components['project'].get_weights()[0], metric, axis=(0, 1, 3))
                norm_energy += (proj_energy / (np.max(proj_energy) + self.cfg.EPSILON_NORM))

            n_prune = int(num_filters * target_ratio)
            if n_prune == 0: continue
                
            n_keep = max(1, num_filters - n_prune)
            keep_indices = np.sort(np.argsort(norm_energy)[-n_keep:])
            
            plans[prefix] = {
                'mask': keep_indices, 
                'original_filters': num_filters,
                'surviving_filters': len(keep_indices),
                'pruned_filters': num_filters - len(keep_indices),
                'ratio': (n_prune / num_filters),
                'norm_energy_mean': float(np.mean(norm_energy)),
                'norm_energy_max': float(np.max(norm_energy))
            }
        return plans

print("[BERHASIL] Cell 3 (Pruning Analyzer Scalable) dimuat.")

[BERHASIL] Cell 3 (Pruning Analyzer Scalable) dimuat.


In [71]:
# ==============================================================================
# [CELL 4] PRUNING SURGEON: PHYSICAL MATRIX REBUILDER
# ==============================================================================
import tensorflow as tf
from typing import Dict, Any

class PruningSurgeon:
    """
    Standar 7 & 9: Modul Pemotongan Fisik Berbasis Functional API.
    Merekonstruksi graf Keras (.get_config) dan mengiris matriks bobot (weights)
    sesuai cetak biru (mask) dari PruningAnalyzer.
    """
    def __init__(self, architecture: ModelArch):
        self.arch = architecture

    def rebuild_and_slice(self, model: tf.keras.Model, plans: Dict[str, Dict[str, Any]]) -> tf.keras.Model:
        """Membuat graf baru dengan filter yang dikurangi, lalu memindahkan bobot spesifik."""
        # 1. REKONSTRUKSI GRAF (CONFIG)
        config = model.get_config()
        for l_conf in config['layers']:
            l_name = l_conf['name']
            l_conf.pop('build_config', None)
            if 'config' in l_conf:
                l_conf['config'].pop('batch_input_shape', None)
                l_conf['config'].pop('input_shape', None)

            matched_prefix = next((p for p in sorted(plans.keys(), key=len, reverse=True) if l_name.startswith(p)), None)
            if matched_prefix:
                p = plans[matched_prefix]
                # Logika Agnostik untuk mengubah dimensi pada Config
                if l_name.endswith(("_expand", "_expand_conv", "se_expand")):
                    l_conf['config']['filters'] = p['surviving_filters']
                elif "squeeze_excite_conv_1" in l_name: # Spesifik MobNet
                    l_conf['config']['filters'] = p['surviving_filters']
                elif "squeeze_excite_conv" in l_name and "conv_1" not in l_name: # Spesifik MobNet
                    old_f = l_conf['config']['filters']
                    l_conf['config']['filters'] = max(1, int(old_f * (1 - p['ratio'])))
                elif "_se_reshape" in l_name: # Spesifik EffNet
                    l_conf['config']['target_shape'] = (1, 1, p['surviving_filters'])

        pruned_model = tf.keras.Model.from_config(config)

        # 2. PEMOTONGAN FISIK MATRIKS (SLICING)
        for new_layer in pruned_model.layers:
            try: old_layer = model.get_layer(new_layer.name)
            except ValueError: continue
            
            if not old_layer.weights: continue
            old_w = old_layer.get_weights()
            
            matched_prefix = next((p for p in sorted(plans.keys(), key=len, reverse=True) if new_layer.name.startswith(p)), None)
            
            # Jika tidak dipruning, salin utuh
            if not matched_prefix or new_layer.weights[0].shape == old_w[0].shape: 
                new_layer.set_weights(old_w)
                continue
                
            mask = plans[matched_prefix]['mask']
            name = new_layer.name
            
            # Eksekusi Pemotongan Berdasarkan Posisi Filter
            if ("_expand" in name or "_se_expand" in name) and "project" not in name and isinstance(new_layer, tf.keras.layers.Conv2D):
                w, b = old_w[0][:, :, :, mask], old_w[1][mask] if len(old_w) > 1 else None
                new_layer.set_weights([w, b] if b is not None else [w])
                
            elif "squeeze_excite_conv_1" in name: # MobNet SE
                in_ch = new_layer.weights[0].shape[2]
                w, b = old_w[0][:, :, :in_ch, mask], old_w[1][mask] if len(old_w) > 1 else None
                if w.shape == new_layer.weights[0].shape: new_layer.set_weights([w, b] if b is not None else [w])
                
            elif "squeeze_excite_conv" in name: # MobNet SE Reduce
                target_out = new_layer.filters
                w, b = old_w[0][:, :, mask, :target_out], old_w[1][:target_out] if len(old_w) > 1 else None
                if w.shape == new_layer.weights[0].shape: new_layer.set_weights([w, b] if b is not None else [w])
                
            elif "_se_reduce" in name: # EffNet SE
                w, b = old_w[0][:, :, mask, :], old_w[1] if len(old_w) > 1 else None
                new_layer.set_weights([w, b] if b is not None else [w])
                
            elif isinstance(new_layer, tf.keras.layers.DepthwiseConv2D):
                w, b = old_w[0][:, :, mask, :], old_w[1][mask] if len(old_w) > 1 else None
                new_layer.set_weights([w, b] if b is not None else [w])
                
            elif ("project" in name and isinstance(new_layer, tf.keras.layers.Conv2D)) or (isinstance(new_layer, tf.keras.layers.BatchNormalization) and "project" not in name):
                if "project_bn" in name: 
                    new_layer.set_weights(old_w)
                elif isinstance(new_layer, tf.keras.layers.BatchNormalization): 
                    new_layer.set_weights([x[mask] for x in old_w])
                else:
                    w, b = old_w[0][:, :, mask, :], old_w[1] if len(old_w) > 1 else None
                    new_layer.set_weights([w, b] if b is not None else [w])
            else:
                new_layer.set_weights(old_w)

        return pruned_model

print("[BERHASIL] Cell 4 (Pruning Surgeon) dimuat.")

[BERHASIL] Cell 4 (Pruning Surgeon) dimuat.


In [72]:
# ==============================================================================
# [CELL 5] STRESS TEST ORCHESTRATOR: THE MAIN ENGINE
# ==============================================================================
import gc
import os
import tensorflow as tf
from tensorflow.keras import models

class StressTestOrchestrator:
    """
    Standar 6 & 8: Konduktor eksperimen One-Shot Stress Test.
    Mengeksekusi siklus pemotongan progresif dan mengevaluasi akurasi.
    Secara dinamis mendeteksi pilihan metrik (L1/L2) dari konfigurasi sentral.
    """
    def __init__(self, config: PruningConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.input_model_path = self.cfg.get_input_model_path(self.arch)
        self.db_path = self.cfg.get_db_path(self.arch)
        
        self.analyzer = PruningAnalyzer(self.cfg, self.arch)
        self.surgeon = PruningSurgeon(self.arch)
        self.logger = PruningSQLiteLogger(self.db_path)
        
        self._build_dataset()

    def _build_dataset(self) -> None:
        """Menyiapkan Validation Dataset khusus untuk testing akurasi Murni (Zero-Shot)."""
        self.val_ds = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_VAL_FFB, image_size=self.cfg.IMG_SIZE, 
            batch_size=self.cfg.VAL_BATCH_SIZE, shuffle=False, label_mode='int', verbose=0
        ).take(1).prefetch(buffer_size=tf.data.AUTOTUNE)

    def run_stress_test(self) -> None:
        print("\n" + "═"*75)
        # Menampilkan metrik yang sedang aktif untuk transparansi eksperimen
        metric_name = self.cfg.PRUNING_METRIC.value.upper()
        print(f"⚙️ MEMULAI STRESS TEST: {self.arch.value} | METRIK: {metric_name}")
        print("═"*75)
        
        # 1. Muat Baseline Model (FP32)
        if not os.path.exists(self.input_model_path):
            raise FileNotFoundError(f"[ERROR] Baseline Model {self.arch.value} tidak ditemukan di: {self.input_model_path}")
            
        print("[INFO] Memuat Model FP32 Baseline...")
        baseline_model = models.load_model(self.input_model_path)
        baseline_model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        orig_params = baseline_model.count_params()

        # 2. Eksekusi Loop Target Rasio (0.0 -> 0.9)
        for target_rate in self.cfg.PRUNING_RATIOS:
            print(f"\n---> MENGUJI RASIO PRUNING: {int(target_rate*100)}%")
            
            if target_rate == 0.0: 
                _, zero_shot_acc = baseline_model.evaluate(self.val_ds, verbose=0)
                entropy = self.analyzer.compute_activation_entropy(baseline_model, self.val_ds)
                new_params = orig_params
                
            else: 
                # IMPLEMENTASI SCALABLE: Injeksi metrik dinamis dari Config
                plans = self.analyzer.plan_normalized(
                    baseline_model, 
                    target_rate, 
                    metric=self.cfg.PRUNING_METRIC
                )
                
                pruned_model = self.surgeon.rebuild_and_slice(baseline_model, plans)
                pruned_model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
                new_params = pruned_model.count_params()
                
                _, zero_shot_acc = pruned_model.evaluate(self.val_ds, verbose=0)
                entropy = self.analyzer.compute_activation_entropy(pruned_model, self.val_ds)
                
                self.logger.log_micro_batch(int(target_rate*100), plans)
                
                del pruned_model
                tf.keras.backend.clear_session()
                gc.collect()

            reduction_pct = ((orig_params - new_params) / float(orig_params)) * 100.0
            print(f"     Parameter Aktif : {new_params:,} (-{reduction_pct:.2f}%)")
            print(f"     Akurasi Murni   : {zero_shot_acc:.4f}")
            print(f"     Feature Entropy : {entropy:.4f}")
            
            self.logger.log_macro({
                'Target_Ratio_Pct': int(target_rate*100),
                'Params_Active': new_params,
                'Params_Reduced_Pct': round(reduction_pct, 2),
                'Zero_Shot_Accuracy': round(zero_shot_acc, 4),
                'Activation_Entropy': round(entropy, 4)
            })

        print(f"\n[BERHASIL] Stress Test {self.arch.value} Selesai. Log terekam di SQLite: {self.db_path}")

print("[BERHASIL] Cell 5 (Stress Test Orchestrator - Scalable) dimuat.")

[BERHASIL] Cell 5 (Stress Test Orchestrator - Scalable) dimuat.


In [73]:
# ==============================================================================
# [CELL 6] STRESS TEST EXECUTOR: BATCH LOOP MODE
# ==============================================================================
import gc
import tensorflow as tf

# 1. Inisialisasi Konfigurasi Global
config = PruningConfig()

# 2. Seleksi Arsitektur
# Berikan tanda pagar (#) pada arsitektur yang tidak ingin diuji saat ini.
MODELS_TO_PRUNE = [
    ModelArch.MOBILENET_V3,
    # ModelArch.EFFICIENTNET_B0,
    # ModelArch.RESNET_50
]

print("\n" + "★"*75)
print(f"🔪 MEMULAI BATCH PRUNING STRESS TEST UNTUK {len(MODELS_TO_PRUNE)} ARSITEKTUR")
print("★"*75)

for idx, arch in enumerate(MODELS_TO_PRUNE, start=1):
    print(f"\n{'='*75}")
    print(f"[{idx}/{len(MODELS_TO_PRUNE)}] EKSEKUSI PRUNING: {arch.value}")
    print(f"{'='*75}")
    
    try:
        # Standar 3 & 6: Reset State & Kunci Seed Spesifik per Iterasi
        tf.keras.backend.clear_session()
        gc.collect()
        set_deterministic_seed(42)
        
        # Eksekusi Orchestrator
        orchestrator = StressTestOrchestrator(config=config, architecture=arch)
        orchestrator.run_stress_test()
        
        # Cleanup Ekstrem pasca-arsitektur untuk menghindari Out of Memory (OOM)
        print(f"\n[CLEANUP] Membersihkan VRAM GPU pasca-Pruning untuk {arch.value}...")
        del orchestrator
        gc.collect()
        
    except FileNotFoundError as e:
        print(f"\n[SKIP] Melompati {arch.value} karena file baseline belum tersedia: {e}")
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Eksekusi Pruning gagal pada {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES ONE-SHOT STRESS TEST SELESAI!")
print("★"*75)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🔪 MEMULAI BATCH PRUNING STRESS TEST UNTUK 1 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

[1/1] EKSEKUSI PRUNING: MobileNetV3Small
[SETUP] Deterministic Seed dikunci pada: 42
[DEBUG PATH] Model ditemukan di struktur Flat: /kaggle/input/models/marrioqqqq/prune-model-ffb/keras/default/1/model_MobileNetV3Small_phase2.keras

═══════════════════════════════════════════════════════════════════════════
⚙️ MEMULAI STRESS TEST: MobileNetV3Small | METRIK: L2_NORM
═══════════════════════════════════════════════════════════════════════════
[INFO] Memuat Model FP32 Baseline...

---> MENGUJI RASIO PRUNING: 0%
     Parameter Aktif : 943,155 (-0.00%)
     Akurasi Murni   : 0.0000
     Feature Entropy : 0.2092

---> MENGUJI RASIO PRUNING: 50%
     Parameter Aktif : 431,071 (-54.29%)
     Akurasi Murni   : 1.0000
     Feature Entropy : 1.9672

[BERHASIL] Stress Test MobileNetV3Small Sel

In [74]:
# ==============================================================================
# [CELL 7] PRUNING VISUALIZER: MACRO (INFO LOSS) & MICRO (L2-NORM) ANALYTICS
# ==============================================================================
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink

class PruningVisualizer:
    """
    Standar 9: Base Class Agnostik untuk merender grafik dari SQLite Pruning Log.
    Menghasilkan Kurva Information Loss (Macro) dan Distribusi L2-Norm (Micro).
    """
    def __init__(self, config: PruningConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.db_path = self.cfg.get_db_path(self.arch)
        
        # Standar 4: Isolasi Folder Visualisasi Khusus
        self.vis_dir = os.path.join(self.cfg.get_output_dir(self.arch), "visualisation")
        os.makedirs(self.vis_dir, exist_ok=True)
        
        # Path File Ekspor
        self.p_macro = os.path.join(self.vis_dir, f"prune_01_{self.arch.value}_macro_info_loss.png")
        self.p_entropy = os.path.join(self.vis_dir, f"prune_02_{self.arch.value}_macro_entropy.png")
        self.p_micro = os.path.join(self.vis_dir, f"prune_03_{self.arch.value}_micro_l2norm.png")
        
        print(f"\n" + "="*60)
        print(f"📊 INISIALISASI VISUALIZER PRUNING: {self.arch.value}")
        print("="*60)
        
        self._validate_database()
        self.df_macro = self._load_data("macro_metrics")
        self.df_micro = self._load_data("micro_metrics")

    def _validate_database(self) -> None:
        if not os.path.exists(self.db_path):
            raise FileNotFoundError(f"[ERROR] Database {self.db_path} tidak ditemukan.")
        if os.path.getsize(self.db_path) == 0:
            raise ValueError(f"[ERROR] Database {self.db_path} kosong.")
            
    def _load_data(self) -> pd.DataFrame:
        """Fungsi helper untuk membaca tabel tertentu dari SQLite."""
        def load(table_name: str) -> pd.DataFrame:
            conn = sqlite3.connect(self.db_path)
            try:
                df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
            except sqlite3.OperationalError:
                df = pd.DataFrame() # Tabel mungkin belum dibuat jika pruning gagal total
            conn.close()
            return df
            
        self._load_data = load
        return load

    # --------------------------------------------------------------------------
    # 1. RENDER MAKRO: KURVA INFORMATION LOSS & PARAMETER DROP
    # --------------------------------------------------------------------------
    def plot_information_loss(self) -> None:
        if self.df_macro.empty:
            print("[WARNING] Data Macro kosong, melewati rendering Information Loss.")
            return

        plt.style.use('default')
        sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
        fig, ax1 = plt.subplots(figsize=(10, 6))

        color_params = '#3498db' if self.arch == ModelArch.MOBILENET_V3 else '#27ae60'
        ax1.set_xlabel('Target Rasio Pruning (%)', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Sisa Parameter Aktif', color=color_params, fontsize=12, fontweight='bold')
        
        # Bar Chart untuk Parameter
        ax1.bar(self.df_macro['target_ratio_pct'].astype(str), self.df_macro['params_active'], 
                color=color_params, alpha=0.6, label='Parameter Aktif')
        ax1.tick_params(axis='y', labelcolor=color_params)

        # Line Chart untuk Akurasi di axis yang berbeda
        ax2 = ax1.twinx()
        color_acc = '#e74c3c' if self.arch == ModelArch.MOBILENET_V3 else '#e67e22'
        ax2.set_ylabel('Zero-Shot Accuracy (%)', color=color_acc, fontsize=12, fontweight='bold')
        ax2.plot(self.df_macro['target_ratio_pct'].astype(str), self.df_macro['zero_shot_accuracy'] * 100, 
                 color=color_acc, marker='o' if self.arch == ModelArch.MOBILENET_V3 else 's', 
                 linestyle='-', linewidth=3, markersize=8, label='Akurasi Murni')
        ax2.tick_params(axis='y', labelcolor=color_acc)

        # Anotasi angka akurasi
        for i, acc in enumerate(self.df_macro['zero_shot_accuracy']):
            ax2.annotate(f"{acc*100:.1f}%", (i, acc*100), textcoords="offset points", xytext=(0,10), 
                         ha='center', fontsize=10, color='darkred', fontweight='bold')

        metric_name = self.cfg.PRUNING_METRIC.value.upper()
        plt.title(f'Kurva Information Loss ({metric_name}) - {self.arch.value}', fontsize=14, fontweight='bold', pad=15)
        fig.tight_layout()
        plt.savefig(self.p_macro, dpi=300, bbox_inches='tight')
        plt.close()

    # --------------------------------------------------------------------------
    # 2. RENDER MAKRO: AKTIVASI ENTROPI
    # --------------------------------------------------------------------------
    def plot_entropy(self) -> None:
        if self.df_macro.empty:
            return
            
        plt.figure(figsize=(10, 4))
        plt.plot(self.df_macro['target_ratio_pct'].astype(str), self.df_macro['activation_entropy'], 
                 color='#8e44ad', marker='D', linewidth=2, markersize=7)
        plt.title(f'Feature Entropy Change - {self.arch.value}', fontsize=14, fontweight='bold', pad=10)
        plt.xlabel('Target Rasio Pruning (%)', fontweight='bold')
        plt.ylabel('Shannon Entropy', fontweight='bold')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(self.p_entropy, dpi=300)
        plt.close()

    # --------------------------------------------------------------------------
    # 3. RENDER MIKRO: DISTRIBUSI NORM ENERGY PER LAYER
    # --------------------------------------------------------------------------
    def plot_micro_layerwise(self) -> None:
        if self.df_micro.empty:
            print("[WARNING] Data Micro kosong, melewati rendering Layerwise Analytics.")
            return

        # Mengambil data pemotongan moderat (misal rasio 50%) sebagai sampel perbandingan
        target_sample = 50 
        df_sample = self.df_micro[self.df_micro['target_ratio_pct'] == target_sample].copy()
        
        if df_sample.empty: # Fallback jika 50% tidak ada
            target_sample = self.df_micro['target_ratio_pct'].max()
            df_sample = self.df_micro[self.df_micro['target_ratio_pct'] == target_sample].copy()

        plt.figure(figsize=(12, 6))
        sns.barplot(data=df_sample, x='block_name', y='l2_norm_energy_mean', color='#34495e')
        
        plt.title(f'Distribusi Energi Norm Rata-rata per Layer (Rasio Uji: {target_sample}%) - {self.arch.value}', 
                  fontsize=14, fontweight='bold', pad=10)
        plt.xlabel('Nama Blok Arsitektur', fontweight='bold')
        plt.ylabel('Rata-rata Energi Filter', fontweight='bold')
        plt.xticks(rotation=90, fontsize=9)
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        
        plt.tight_layout()
        plt.savefig(self.p_micro, dpi=300)
        plt.close()

    def run_all_visualisations(self) -> None:
        self.plot_information_loss()
        self.plot_entropy()
        self.plot_micro_layerwise()
        
        print(f"[BERHASIL] Gambar Resolusi Tinggi Tersimpan di: {self.vis_dir}")
        display(FileLink(self.p_macro))
        display(FileLink(self.p_entropy))
        display(FileLink(self.p_micro))

print("[BERHASIL] Cell 7 (Class PruningVisualizer) dimuat.")

[BERHASIL] Cell 7 (Class PruningVisualizer) dimuat.


In [75]:
# ==============================================================================
# [CELL 8] PRUNING VISUALIZATION EXECUTOR (BATCH RUNNER)
# ==============================================================================

# Menggunakan MODELS_TO_PRUNE dari Cell 6 sebelumnya jika ada.
# Jika cell dieksekusi terpisah, gunakan fallback arsitektur.
try:
    models_to_visualize = MODELS_TO_PRUNE
except NameError:
    models_to_visualize = [ModelArch.MOBILENET_V3] 

print("\n" + "★"*75)
print(f"🎨 MEMULAI RENDER VISUALISASI PRUNING UNTUK {len(models_to_visualize)} ARSITEKTUR")
print("★"*75)

config_vis = PruningConfig()

for arch in models_to_visualize:
    try:
        # Inisialisasi visualizer untuk arsitektur spesifik
        visualizer = PruningVisualizer(config=config_vis, architecture=arch)
        
        # Render dan simpan semua metrik (Macro & Micro)
        visualizer.run_all_visualisations()
        
    except FileNotFoundError as e:
        print(e)
    except ValueError as e:
        print(e)
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Gagal merender grafik untuk {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES RENDER GRAFIK PRUNING SELESAI!")
print("★"*75)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🎨 MEMULAI RENDER VISUALISASI PRUNING UNTUK 1 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

📊 INISIALISASI VISUALIZER PRUNING: MobileNetV3Small

[CRITICAL ERROR] Gagal merender grafik untuk MobileNetV3Small: PruningVisualizer._load_data() takes 1 positional argument but 2 were given

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 SELURUH PROSES RENDER GRAFIK PRUNING SELESAI!
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [76]:
# ==============================================================================
# [CELL 9] DEBUG: INSPEKSI STRUKTUR DIREKTORI OUTPUT PRUNING
# ==============================================================================
import os

def print_directory_tree(startpath: str) -> None:
    print(f"\n🌳 STRUKTUR DIREKTORI PRUNING: {startpath}")
    print("="*60)
    
    if not os.path.exists(startpath):
        print(f"[WARNING] Direktori {startpath} belum dibuat.")
        return

    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}📂 {os.path.basename(root)}/")
        
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            if f.endswith('.db'): icon = "🗄️"
            elif f.endswith('.png'): icon = "🖼️"
            else: icon = "📄"
            print(f"{subindent}{icon} {f}")

config_debug = PruningConfig()
print_directory_tree(config_debug.BASE_OUTPUT_DIR)


🌳 STRUKTUR DIREKTORI PRUNING: /kaggle/working/pruning_outputs
📂 pruning_outputs/
    📂 MobileNetV3Small/
        🗄️ pruning_stresstest_MobileNetV3Small.db
        📂 visualisation/
